In [28]:
import boto3
import os
import tarfile
import sagemaker
from sagemaker.train.configs import InputData
from sagemaker.train import ModelTrainer
from sagemaker.core import image_uris
from sagemaker.core.training.configs import (
    Compute,
    OutputDataConfig,
    StoppingCondition,
    InputData
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [29]:
sts = boto3.client("sts")
s3 = boto3.client("s3")

identity = sts.get_caller_identity()

print(identity["Arn"])

arn:aws:sts::066401718601:assumed-role/AmazonSageMaker-ExecutionRole-20260820T174073/SageMaker


In [30]:
execution_role = (
    "arn:aws:iam::066401718601:role/"
    "service-role/AmazonSageMaker-ExecutionRole-20260820T174073"
)


In [31]:
#definir rutas a s3
bucket = "telco-constumer-churn-066401718601-us-east-2-an"

train_s3 = f"s3://{bucket}/Processed/train.csv"
validation_s3 = f"s3://{bucket}/Processed/val.csv"
test_s3 = f"s3://{bucket}/Processed/test.csv"
output_s3 = f"s3://{bucket}/Models/"

print(train_s3)
print(validation_s3)
print(test_s3)

s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/train.csv
s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/val.csv
s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/test.csv


In [32]:
print(ModelTrainer)
print(InputData)

<class 'sagemaker.train.model_trainer.ModelTrainer'>
<class 'sagemaker.core.training.configs.InputData'>


In [33]:
#configuracion de entrenamiento, calculos, ruta de salida y condicion de pausa
compute = Compute(
    instance_type="ml.m5.large",
    instance_count=1
)

output_data_config = OutputDataConfig(
    s3_output_path=output_s3
)

stopping_condition = StoppingCondition(
    max_runtime_in_seconds=900
)

In [34]:
#se definen los datos de entrada
train_input = InputData(
    channel_name="train",
    data_source=train_s3,
    content_type="text/csv"
)

validation_input = InputData(
    channel_name="validation",
    data_source=validation_s3,
    content_type="text/csv"
)

In [35]:
#crear imagen de xgboost
region = boto3.Session().region_name

xgboost_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="training"
)

print("XGBoost image:")
print(xgboost_image)

[09/14/26 18:18:17] INFO     Ignoring unnecessary Python version: py3.                            ]8;id=7507848;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=7507849;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#608\608]8;;\

                    INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=7507854;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=7507855;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

XGBoost image:
257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.7-1


In [36]:
#crear el model trainer con Xgboost
trainer = ModelTrainer(
    role=execution_role,
    base_job_name="telco-churn-xgboost",
    training_image=xgboost_image,
    compute=compute,
    stopping_condition=stopping_condition,
    output_data_config=output_data_config,
    input_data_config=[
        train_input,
        validation_input
    ],
    hyperparameters={
        "objective": "binary:logistic",
        "num_round": 100,
        "max_depth": 5,
        "eta": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "eval_metric": "auc"
    }
)

[09/14/26 18:18:20] INFO     SageMaker session not provided. Using default Session.                  ]8;id=7507860;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=7507861;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=7507866;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=7507867;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=7507872;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=7507873;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.                     
                             7-1                                                                                   

In [10]:
training_job = trainer.train()
print(training_job)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=7507054;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7507055;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=7507062;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7507063;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=7507070;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=7507071;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

Output()

...

                    INFO     telco-churn-xgboost-20260914181248/algo-1-1789409612:               ]8;id=7507815;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7507816;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [97]#011train-auc:0.96087#011validation-auc:0.84637                                   

                    INFO     telco-churn-xgboost-20260914181248/algo-1-1789409612:               ]8;id=7507821;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7507822;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [98]#011train-auc:0.96108#011validation-auc:0.84672                                   

                    INFO     telco-churn-xgboost-20260914181248/algo-1-1789409612:               ]8;id=7507827;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7507828;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [99]#011train-auc:0.96145#011validation-auc:0.84652                                   

[09/14/26 18:15:51] INFO     Final Resource Status: Completed                                    ]8;id=7507834;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7507835;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

None


In [37]:
#encontrar donde se guardo el modelo
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="Models/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

Models/telco-churn-xgboost-20260913193637/output/model.tar.gz
Models/telco-churn-xgboost-20260914175813/output/model.tar.gz
Models/telco-churn-xgboost-20260914181248/output/model.tar.gz


In [38]:
#descargar modelo
model_key = "Models/telco-churn-xgboost-20260914175813/output/model.tar.gz"

os.makedirs("model", exist_ok=True)

s3.download_file(
    bucket,
    model_key,
    "model/model.tar.gz"
)

print("✅ Modelo descargado")

✅ Modelo descargado


In [39]:
#extraer modelo
with tarfile.open("model/model.tar.gz", "r:gz") as tar:
    tar.extractall("model")

print(os.listdir("model"))

['model.tar.gz', 'xgboost-model']


/tmp/ipykernel_1091/969497294.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("model")


In [40]:
#cargar modelo
import xgboost as xgb

model = xgb.Booster()

model.load_model("model/xgboost-model")

print("✅ Modelo cargado")

✅ Modelo cargado


In [41]:
#descargar test
import pandas as pd
response = s3.get_object(
    Bucket=bucket,
    Key="Processed/test.csv"
)

test = pd.read_csv(response["Body"])

In [42]:
#separar x y Y
y_test = test["Churn"]
X_test = test.drop(columns=["Churn"])

In [43]:
#predecir
dtest = xgb.DMatrix(X_test)

y_pred_proba = model.predict(dtest)

print(y_pred_proba[:10])

[0.13193327 0.11406007 0.32247064 0.69396806 0.11251987 0.1665309
 0.50534546 0.7266862  0.09711456 0.21823052]


In [44]:
y_pred = (y_pred_proba >= 0.5).astype(int)

print(y_pred[:20])

[0 0 0 1 0 0 1 1 0 0 0 0 1 1 0 0 0 1 1 0]


In [45]:
#evaluamos modelo a travez de diversas metricas
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.8491284249179757
Accuracy: 0.7880512091038406
Precision: 0.5600961538461539
Recall: 0.6695402298850575
F1: 0.6099476439790575

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.83      0.85      1058
           1       0.56      0.67      0.61       348

    accuracy                           0.79      1406
   macro avg       0.72      0.75      0.73      1406
weighted avg       0.80      0.79      0.79      1406


Confusion Matrix:
[[875 183]
 [115 233]]
